# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

# Chương 9: Mô hình ngôn ngữ đã tiền huấn luyện (kiểu BERT)

Trong chương này, bạn sẽ sử dụng mô hình ngôn ngữ đã được huấn luyện sẵn theo kiến trúc BERT để thực hiện:

- dự đoán từ bị che (masked language modeling)
- tạo vector biểu diễn câu
- xây dựng bộ phân loại cảm xúc (positive/negative)

## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

## 80. Tokenization

Hãy tách câu sau thành các token:

```
“The movie was full of incomprehensibilities.”
```
và hiển thị dãy token tương ứng.

In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = "The movie was full of incomprehensibilities."
doc = nlp(text)
tokens = [token.text for token in doc]
ids = [token.idx for token in doc]

print(tokens)
print(ids)

['The', 'movie', 'was', 'full', 'of', 'incomprehensibilities', '.']
[0, 4, 10, 14, 19, 22, 43]


In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "The movie was full of incomprehensibilities."
tokens = tokenizer.tokenize(text)

print(tokens)
print(" ".join(tokens))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

['the', 'movie', 'was', 'full', 'of', 'inc', '##omp', '##re', '##hen', '##si', '##bilities', '.']
the movie was full of inc ##omp ##re ##hen ##si ##bilities .


## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

## 81. Dự đoán từ bị che (Masked Token Prediction)

Cho câu:

```
“The movie was full of [MASK].”
```

Hãy dự đoán:

- token phù hợp nhất để điền vào vị trí [MASK]

In [3]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

text = "The movie was full of [MASK]."
inputs = tokenizer(text, return_tensors="pt")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

In [5]:
mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]

In [6]:
mask_logits = logits[0, mask_token_index, :]

In [7]:
top_id = torch.argmax(mask_logits).item()
prediction = tokenizer.decode([top_id])

print("Predicted token:", prediction)

Predicted token: fun


## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

## 82. Top-k dự đoán cho masked token

Với câu:

```
“The movie was full of [MASK].”
```

Hãy tìm:

- 10 token có xác suất cao nhất
- kèm xác suất (likelihood) tương ứng

In [8]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

text = "The movie was full of [MASK]."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]

mask_logits = logits[0, mask_token_index, :]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
top_k = torch.topk(mask_logits, k=10)

top_ids = top_k.indices.tolist()[0]
top_scores = top_k.values.tolist()[0]

top_tokens = [tokenizer.decode([i]).strip() for i in top_ids]

for token, score in zip(top_tokens, top_scores):
    print(f"{token} | {score}")

fun | 9.28895092010498
surprises | 8.8098783493042
drama | 8.414626121520996
stars | 7.9188551902771
laughs | 7.850265979766846
action | 7.586291313171387
excitement | 7.561453342437744
people | 7.5213751792907715
tension | 7.325096607208252
music | 7.299192905426025


In [10]:
probs = torch.softmax(mask_logits, dim=-1)
top_k = torch.topk(probs, k=10)

top_ids = top_k.indices.tolist()[0]
top_probs = top_k.values.tolist()[0]

top_tokens = [tokenizer.decode([i]).strip() for i in top_ids]

for token, prob in zip(top_tokens, top_probs):
    print(f"{token} | {prob:.6f}")

fun | 0.107119
surprises | 0.066345
drama | 0.044684
stars | 0.027217
laughs | 0.025413
action | 0.019517
excitement | 0.019038
people | 0.018290
tension | 0.015031
music | 0.014646


## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

## 83. Vector câu bằng token [CLS]

Với các câu sau:

- “The movie was full of fun.”
- “The movie was full of excitement.”
- “The movie was full of crap.”
- “The movie was full of rubbish.”


Hãy:

- lấy embedding của token [CLS] từ lớp cuối của BERT
- tính cosine similarity giữa tất cả các cặp câu

In [11]:
from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
vectors = []

with torch.no_grad():
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors="pt")
        outputs = model(**inputs)

        cls_vec = outputs.last_hidden_state[:, 0, :].squeeze(0)
        vectors.append(cls_vec)

In [13]:
for i in range(len(vectors)):
    for j in range(i + 1, len(vectors)):
        sim = F.cosine_similarity(vectors[i], vectors[j], dim=0)
        print(f"{i} - {j}: {sim.item():.4f}")

0 - 1: 0.9881
0 - 2: 0.9558
0 - 3: 0.9475
1 - 2: 0.9541
1 - 3: 0.9487
2 - 3: 0.9807


## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

## 84. Vector câu bằng trung bình embedding

Tương tự bài 83, nhưng thay vì dùng [CLS], hãy:

- lấy trung bình embedding của tất cả token trong câu
- tính cosine similarity giữa các câu

In [14]:
from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
vectors = []

with torch.no_grad():
    for sent in sentences:
        inputs = tokenizer(sent, return_tensors="pt")
        outputs = model(**inputs)

        token_vecs = outputs.last_hidden_state.squeeze(0)
        attention_mask = inputs["attention_mask"][0].unsqueeze(-1)
        masked_vecs = token_vecs * attention_mask

        mean_vec = masked_vecs.sum(dim=0) / attention_mask.sum()
        vectors.append(mean_vec)

In [16]:
for i in range(len(vectors)):
    for j in range(i + 1, len(vectors)):
        sim = F.cosine_similarity(vectors[i], vectors[j], dim=0)
        print(f"{i} - {j}: {sim.item():.4f}")

0 - 1: 0.9568
0 - 2: 0.8490
0 - 3: 0.8169
1 - 2: 0.8352
1 - 3: 0.7938
2 - 3: 0.9226


## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) から訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、さらに全てのテキストはトークン列に変換せよ。

## 85. Chuẩn bị dataset

Tải dataset:

- GLUE
- SST-2

Hãy:

- đọc train.tsv và dev.tsv
- chuyển toàn bộ câu thành token sequence bằng BERT tokenizer

In [21]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

--2026-06-18 07:02:46--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.81, 65.9.168.52, 65.9.168.4, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.81|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip’

SST-2.zip           100%[===================>]   7.09M  --.-KB/s    in 0.05s   

2026-06-18 07:02:46 (142 MB/s) - ‘SST-2.zip’ saved [7439277/7439277]

Archive:  SST-2.zip
   creating: SST-2/
  inflating: SST-2/dev.tsv           
   creating: SST-2/original/
  inflating: SST-2/original/README.txt  
  inflating: SST-2/original/SOStr.txt  
  inflating: SST-2/original/STree.txt  
  inflating: SST-2/original/datasetSentences.txt  
  inflating: SST-2/original/datasetSplit.txt  
  inflating: SST-2/original/dictionary.txt  
  inflating: SST-2/original/original_rt_snippets.txt  
  inflating: SST-2/original/sentiment_l

In [23]:
import pandas as pd

train_df = pd.read_csv("SST-2/train.tsv",sep="\t")
dev_df = pd.read_csv("SST-2/dev.tsv",sep="\t")

print(len(train_df))
print(len(dev_df))

67349
872


In [24]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [25]:
import torch

def encode_example(text, label):
    encoded = tokenizer(text, truncation=True)

    return {
        "text": text,
        "label": torch.tensor([float(label)]),
        "input_ids": torch.tensor(
            encoded["input_ids"]
        ),
        "attention_mask": torch.tensor(
            encoded["attention_mask"]
        )
    }

In [26]:
train_data = []
dev_data = []

for _, row in train_df.iterrows():

    sample = encode_example(
        row["sentence"],
        row["label"]
    )

    train_data.append(sample)

for _, row in dev_df.iterrows():

    sample = encode_example(
        row["sentence"],
        row["label"]
    )

    dev_data.append(sample)

print(train_data[0])
print(dev_data[0])

{'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}
{'text': "it 's a charming and often affecting journey . ", 'label': tensor([1.]), 'input_ids': tensor([  101,  2009,  1005,  1055,  1037, 11951,  1998,  2411, 12473,  4990,
         1012,   102]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


## 86. ミニバッチの作成

85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

## 86. Tạo mini-batch

Từ dữ liệu ở bài 85:

- chọn một phần nhỏ (ví dụ 4 sample đầu)
- thực hiện padding
- đưa về cùng độ dài
- tạo mini-batch tensor

In [27]:
batch = train_data[:4]

max_len = max(len(sample["input_ids"]) for sample in batch)

padded_ids = []
padded_masks = []
labels = []

for sample in batch:
    ids = sample["input_ids"]
    mask = sample["attention_mask"]

    pad_len = max_len - len(ids)

    ids = torch.cat([
        ids,
        torch.zeros(
            pad_len,
            dtype=torch.long
        )
    ])

    mask = torch.cat([
        mask,
        torch.zeros(
            pad_len,
            dtype=torch.long
        )
    ])

    padded_ids.append(ids)
    padded_masks.append(mask)

    labels.append(sample["label"])

In [28]:
batch_dict = {
    "input_ids": torch.stack(padded_ids),
    "attention_mask": torch.stack(padded_masks),
    "label": torch.stack(labels)
}

print(batch_dict["input_ids"].shape)

torch.Size([4, 15])


In [29]:
# version 2
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    input_ids = pad_sequence(
        [x["input_ids"] for x in batch],
        batch_first=True,
        padding_value=0
    )

    attention_mask = pad_sequence(
        [x["attention_mask"] for x in batch],
        batch_first=True,
        padding_value=0
    )

    labels = torch.stack(
        [x["label"] for x in batch]
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "label": labels
    }

In [32]:
batch = collate_fn(train_data[:4])

print(batch)

{'input_ids': tensor([[  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
             0,     0,     0,     0,     0],
        [  101,  3397,  2053, 15966,  1010,  2069,  4450,  2098, 18201,  2015,
           102,     0,     0,     0,     0],
        [  101,  2008,  7459,  2049,  3494,  1998, 10639,  2015,  2242,  2738,
          3376,  2055,  2529,  3267,   102],
        [  101,  3464, 12580,  8510,  2000,  3961,  1996,  2168,  2802,   102,
             0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]), 'label': tensor([[0.],
        [0.],
        [1.],
        [0.]])}


## 87. ファインチューニング

訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

## 87. Fine-tuning

Hãy fine-tune mô hình BERT đã pretrain:

- trên tập train SST-2
- cho bài toán phân loại sentiment

Sau đó:

- đo accuracy trên dev set

In [38]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [39]:
import torch
from transformers import BertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_data,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [43]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [45]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = (
            batch["label"]
            .squeeze(1)
            .long()
            .to(device)
        )

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1} | Loss = {avg_loss:.4f}"
    )

Epoch 1 | Loss = 0.1253
Epoch 2 | Loss = 0.0806
Epoch 3 | Loss = 0.0578


In [51]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = (
            batch["label"]
            .squeeze(1)
            .long()
            .to(device)
        )

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Dev Accuracy: {accuracy:.4f}")

Dev Accuracy: 0.9289


## 88. 極性分析

問題87でファインチューニングされたモデルを用いて、以下の文の極性を予測せよ。

- "The movie was full of incomprehensibilities."
- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

## 88. Phân tích cảm xúc

Dùng mô hình đã fine-tune ở bài 87 để dự đoán sentiment cho các câu:

- “The movie was full of incomprehensibilities.”
- “The movie was full of fun.”
- “The movie was full of excitement.”
- “The movie was full of crap.”
- “The movie was full of rubbish.”


In [52]:
sentences = [
    "The movie was full of incomprehensibilities.",
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]


In [53]:
model.eval()

with torch.no_grad():
    for sentence in sentences:
        inputs = tokenizer(sentence, return_tensors="pt")

        outputs = model(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device)
        )

        probs = torch.softmax(outputs.logits, dim=1)[0]

        print(sentence)
        print(f"Negative: {probs[0]:.4f}")
        print(f"Positive: {probs[1]:.4f}")
        print()

The movie was full of incomprehensibilities.
Negative: 0.9981
Positive: 0.0019

The movie was full of fun.
Negative: 0.0002
Positive: 0.9998

The movie was full of excitement.
Negative: 0.0005
Positive: 0.9995

The movie was full of crap.
Negative: 0.9995
Positive: 0.0005

The movie was full of rubbish.
Negative: 0.9996
Positive: 0.0004



## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャ（例えば[CLS]トークンを用いるか、各トークンの最大値プーリングを用いるなど）の分類モデルを設計し、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

## 89. Thay đổi kiến trúc

Thiết kế một kiến trúc khác với bài 87, ví dụ:

- dùng token [CLS]
- hoặc dùng max pooling trên token embeddings

Sau đó:

- fine-tune lại mô hình
- đánh giá accuracy trên dev set

### Mean Pooling

In [61]:
import torch
import torch.nn as nn
from transformers import BertModel

class BertMeanPoolClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        hidden_states = outputs.last_hidden_state

        pooled = hidden_states.mean(dim=1)

        logits = self.classifier(pooled)

        return logits

In [62]:
model = BertMeanPoolClassifier().to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

criterion = nn.CrossEntropyLoss()

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = (batch["input_ids"].to(device))

        attention_mask = (batch["attention_mask"].to(device))

        labels = (batch["label"].squeeze(1).long().to(device))

        logits = model(input_ids,attention_mask)

        loss = criterion(logits,labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (total_loss / len(train_loader))

    print(f"Epoch {epoch+1} | Loss = {avg_loss:.4f}")

Epoch 1 | Loss = 0.2001


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in dev_loader:
        input_ids = (batch["input_ids"].to(device))

        attention_mask = (batch["attention_mask"].to(device))

        labels = (batch["label"].squeeze(1).long().to(device))

        logits = model(input_ids, attention_mask)

        preds = torch.argmax(logits,dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Dev Accuracy: {accuracy:.4f}")

### MaxPooling

In [54]:
import torch
import torch.nn as nn

from transformers import BertModel

class BertMaxPoolClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        self.classifier = nn.Linear(768,2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        hidden_states = outputs.last_hidden_state

        pooled = torch.max(hidden_states, dim=1).values

        logits = self.classifier(pooled)

        return logits

In [56]:
model = BertMaxPoolClassifier().to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

criterion = nn.CrossEntropyLoss()

In [58]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = (batch["input_ids"].to(device))

        attention_mask = (batch["attention_mask"].to(device))

        labels = (batch["label"].squeeze(1).long().to(device))

        logits = model(input_ids,attention_mask)

        loss = criterion(logits,labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (total_loss / len(train_loader))

    print(f"Epoch {epoch+1} | Loss = {avg_loss:.4f}")

Epoch 1 | Loss = 0.2034
Epoch 2 | Loss = 0.1049
Epoch 3 | Loss = 0.0695


In [59]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in dev_loader:
        input_ids = (batch["input_ids"].to(device))

        attention_mask = (batch["attention_mask"].to(device))

        labels = (batch["label"].squeeze(1).long().to(device))

        logits = model(input_ids, attention_mask)

        preds = torch.argmax(logits,dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Dev Accuracy: {accuracy:.4f}")

Dev Accuracy: 0.9289


### CLS + Hidden Layer

In [ ]:
class BertMLPClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        self.fc1 = nn.Linear(768,50)

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(50,2)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        cls = outputs.last_hidden_state[:, 0, :]

        x = self.fc1(cls)

        x = self.relu(x)

        logits = self.fc2(x)

        return logits

In [ ]:
model = BertMLPClassifier().to(device)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

criterion = nn.CrossEntropyLoss()

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:
        input_ids = (batch["input_ids"].to(device))

        attention_mask = (batch["attention_mask"].to(device))

        labels = (batch["label"].squeeze(1).long().to(device))

        logits = model(input_ids,attention_mask)

        loss = criterion(logits,labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (total_loss / len(train_loader))

    print(f"Epoch {epoch+1} | Loss = {avg_loss:.4f}")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in dev_loader:
        input_ids = (batch["input_ids"].to(device))

        attention_mask = (batch["attention_mask"].to(device))

        labels = (batch["label"].squeeze(1).long().to(device))

        logits = model(input_ids, attention_mask)

        preds = torch.argmax(logits,dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(f"Dev Accuracy: {accuracy:.4f}")